# FarmPulse - MLOps tracking

Notebook de démonstration: DVC pour les données, MLflow pour les runs/artifacts, ONNX pour le déploiement, Docker et CI pour la reproductibilité.

In [1]:
from pathlib import Path
import sys, json, subprocess, mlflow

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(subprocess.run(['dvc', 'status'], cwd=ROOT, capture_output=True, text=True).stdout)
print((ROOT / 'params.yaml').read_text())

export_densenet:
	changed deps:
		modified:           models\densenet121_best.pt
		modified:           src\models\model.py
		modified:           src\models\export_onnx.py
	changed outs:
		deleted:            models\densenet121.onnx

data:
  raw_dir: data/raw
  processed_dir: data/processed
  image_size: 224
  validation_size: 0.15
  test_size: 0.15
  seed: 42
model:
  name: efficientnet_b0
  pretrained: true
  dropout: 0.3
training:
  batch_size: 32
  epochs: 10
  learning_rate: 0.0003
  weight_decay: 0.0001
  num_workers: 0



In [2]:
from src.utils.tracking import mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri(ROOT))
client = mlflow.MlflowClient()
experiments = client.search_experiments()
[(experiment.name, experiment.experiment_id) for experiment in experiments]

[('Default', '0')]

In [3]:
runs = client.search_runs(experiment_ids=[e.experiment_id for e in experiments], max_results=20, order_by=['attribute.start_time DESC']) if experiments else []
[(run.info.run_id, run.info.run_name, run.data.metrics) for run in runs]

[('1c5908f4b8604dda94597a65ebd453a0',
  'baseline_cnn-test-evaluation',
  {'accuracy': 0.8109854604200323, 'f1_weighted': 0.8073852203622388}),
 ('04dccef43dd0400ba9cea295f7a7ff3b',
  'baseline_cnn',
  {'train_loss': 0.6371806766809446,
   'accuracy': 0.8161550888529887,
   'f1_weighted': 0.8132521668006157,
   'training_time_seconds': 5688.255448900018}),
 ('75c9a8f63a05409fb3e207ef6c173af9',
  'efficientnet_b0-val-evaluation',
  {'accuracy': 0.9961227786752828, 'f1_weighted': 0.996122775623225}),
 ('fae6c0672c394922889d6abbe8d99046',
  'efficientnet_b0-val-evaluation',
  {'accuracy': 0.9961227786752828, 'f1_weighted': 0.996122775623225}),
 ('2dc31e1bcf2641e0bf9109574d011393',
  'efficientnet_b0-val-evaluation',
  {'accuracy': 0.9961227786752828, 'f1_weighted': 0.996122775623225}),
 ('c8387fa541d543c6a38aa0bd1e7765fb',
  'densenet121-val-evaluation',
  {'accuracy': 0.9951534733441034, 'f1_weighted': 0.9951535011226467}),
 ('3edb5da886484c7ba27593ee0bac968f',
  'densenet121',
  {'train

In [ ]:
checkpoint = ROOT / 'models' / 'best.pt'
if checkpoint.exists():
    from src.models.export_onnx import export
    export(checkpoint, ROOT / 'models' / 'model.onnx')
    print('ONNX exported')
else:
    print('Train first: models/best.pt does not exist yet')

## Commandes de livraison

```powershell
dvc repro
pytest backend/tests -q
docker compose up --build
```

Pour la soutenance, montrer l'historique MLflow, `dvc status`, le build CI et l'inférence Flutter.